### BTS DB1B

In [3]:
import sys
import os

# Check Python version
print(f"Python Version: `{sys.version}`")  # Detailed version info
print(f"Base Python location: `{sys.base_prefix}`")
print(f"Current Environment location: `{os.path.basename(sys.prefix)}`", end='\n\n')

Python Version: `3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]`
Base Python location: `C:\Users\LMT\AppData\Local\Programs\Python\Python312`
Current Environment location: `.venv`



In [4]:
sys.path.append("../src") if "../src" not in sys.path else None

In [5]:
import pandas as pd
from pathlib import Path
import logging

import data_handler

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)

#### Data Management with `BTSDataHandler`

In [6]:
handler = data_handler.BTSDataHandler(base_dir=Path("data/bts"))

##### Download CSVs from BTS website (separate subtask)

In [13]:
# Download all quarters already identified in the project (2024 Q1 - 2025 Q2)
results = handler.download_range(
    start=data_handler.Quarter(2024, 1),
    end=data_handler.Quarter(2025, 2),
)

# Check what's available locally
inventory = handler.list_available_quarters()
for stage, quarters in inventory.items():
    print(f"{stage}: {[str(q) for q in quarters]}")


2026-03-28 23:47:34,242 [INFO] Downloading 6 quarters: 2024_Q1 to 2025_Q2
2026-03-28 23:47:39,269 [INFO] Download complete: 6/6 succeeded (11677.4 MB total)


raw: []
csv: ['2024_Q1', '2024_Q2', '2024_Q3', '2024_Q4', '2025_Q1', '2025_Q2']
parquet: ['2024_Q1', '2024_Q2', '2024_Q3', '2024_Q4', '2025_Q1', '2025_Q2']
processed: []


In [ ]:
# Free disk space after Parquet conversion (separate subtask)
handler.cleanup_zips()


##### Convert CSVs to Parquet and verify integrity

In [16]:
# Convert all downloaded quarters
results = handler.convert_range_to_parquet(
    start=data_handler.Quarter(2024, 1),
    end=data_handler.Quarter(2025, 2),
    delete_csv=False,  # keep CSVs until verified
)

# Print summary table
print(f"\n{'Quarter':<12} {'Rows':>12} {'CSV MB':>10} {'Parquet MB':>12} {'Ratio':>8}")
print("-" * 58)
for r in results:
    if r.success:
        print(
            f"{str(r.quarter):<12} {r.row_count:>12,} {r.csv_size_mb:>10.1f} "
            f"{r.parquet_size_mb:>12.1f} {r.compression_ratio:>7.1f}x"
        )
    else:
        print(f"{str(r.quarter):<12} FAILED: {r.error}")

# Verify a sample quarter
info = handler.verify_parquet(data_handler.Quarter(2024, 1))
for k, v in info.items():
    print(f"  {k}: {v}")


2026-03-28 23:54:39,912 [INFO] Converting 6 quarters to Parquet
Converting quarters:   0%|          | 0/6 [00:00<?, ?quarter/s]2026-03-28 23:54:39,923 [INFO] Parquet already exists, skipping: data\bts\parquet\db1b_market_2024_1.parquet (7403594 rows, 111.9 MB)
2026-03-28 23:54:39,929 [INFO] Parquet already exists, skipping: data\bts\parquet\db1b_market_2024_2.parquet (8527138 rows, 129.4 MB)
2026-03-28 23:54:39,932 [INFO] Parquet already exists, skipping: data\bts\parquet\db1b_market_2024_3.parquet (8290377 rows, 126.8 MB)
2026-03-28 23:54:39,935 [INFO] Parquet already exists, skipping: data\bts\parquet\db1b_market_2024_4.parquet (8518002 rows, 128.6 MB)
2026-03-28 23:54:39,939 [INFO] Parquet already exists, skipping: data\bts\parquet\db1b_market_2025_1.parquet (7290528 rows, 111.6 MB)
2026-03-28 23:54:39,945 [INFO] Parquet already exists, skipping: data\bts\parquet\db1b_market_2025_2.parquet (8442941 rows, 129.9 MB)
Converting quarters: 100%|██████████| 6/6 [00:00<00:00, 178.50quarter


Quarter              Rows     CSV MB   Parquet MB    Ratio
----------------------------------------------------------
2024_Q1         7,403,594        0.0        111.9     0.0x
2024_Q2         8,527,138        0.0        129.4     0.0x
2024_Q3         8,290,377        0.0        126.8     0.0x
2024_Q4         8,518,002        0.0        128.6     0.0x
2025_Q1         7,290,528        0.0        111.6     0.0x
2025_Q2         8,442,941        0.0        129.9     0.0x
  exists: True
  quarter: 2024_Q1
  num_rows: 7403594
  num_columns: 30
  num_row_groups: 15
  size_mb: 111.93181133270264
  columns: ['ItinID', 'MktID', 'MktCoupons', 'Year', 'Quarter', 'OriginAirportID', 'Origin', 'OriginCountry', 'OriginState', 'DestAirportID', 'Dest', 'DestCountry', 'DestState', 'AirportGroup', 'TkCarrierChange', 'TkCarrierGroup', 'OpCarrierChange', 'OpCarrierGroup', 'RPCarrier', 'TkCarrier', 'OpCarrier', 'BulkFare', 'Passengers', 'MktFare', 'MktDistance', 'MktDistanceGroup', 'MktMilesFlown', 'NonStop

In [ ]:
# Once verified, delete CSVs to reclaim ~9 GB of disk space
# handler.cleanup_csvs()

##### Batch processing test

How Batch Processing Fits the Pipeline

```
                    24 GB RAM Boundary
                    ─────────────────────
                    │                   │
  Parquet Files     │   iter_quarters   │    Output
  ─────────────     │   ─────────────   │    ──────
  2024_Q1.parquet ──┤                   ├──> Aggregates
  2024_Q2.parquet ──┤  One quarter at   ├──> Samples
  2024_Q3.parquet ──┤  a time in RAM    ├──> Route extracts
  2024_Q4.parquet ──┤                   ├──> Feature tables
  2025_Q1.parquet ──┤  ~800 MB each     │
  2025_Q2.parquet ──┤                   │
                    │                   │
                    ─────────────────────
```

The `iter_quarters` generator is the foundation -- `batch_aggregate`, `get_route_data`, and `create_sample` all delegate to it. This guarantees that multi-quarter operations never exceed one quarter of memory at a time (~800 MB per quarter with all columns, much less with column selection).

In [15]:
q1 = data_handler.Quarter(2024, 1)

# 1. Check memory estimate before loading
est = handler.memory_estimate(q1)
for k, v in est.items():
    print(f"  {k}: {v}")

# 2. Create a 10% sample for fast local iteration
sample_path = handler.create_sample(q1, q1, fraction=0.1)

# 3. Load sample with specific columns for exploration
df = handler.load_sample(columns=[
    "Origin", "Dest", "MktFare", "MktDistance",
    "RPCarrier", "Passengers", "MktCoupons",
    "Year", "Quarter",
])

# 4. Quick sanity checks
print(f"\nShape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nMktFare stats:")
print(df["MktFare"].describe())
print(f"\nTop 10 routes:")
print(
    df.groupby(["Origin", "Dest"])["MktFare"]
    .agg(["mean", "count"])
    .sort_values("count", ascending=False)
    .head(10)
)
print(f"\nCarriers: {df['RPCarrier'].nunique()}")
print(f"Airports (origin): {df['Origin'].nunique()}")

# 5. Route-specific extraction (useful for later comparison with UK routes)
hub_data = handler.get_route_data(
    q1, q1,
    origins=["JFK", "LAX", "ORD", "ATL", "DFW"],
    columns=[
        "Origin", "Dest", "MktFare", "MktDistance",
        "RPCarrier", "Passengers", "MktCoupons",
    ],
)
print(f"\nMajor hub data: {len(hub_data)} rows")


  quarter: 2024_Q1
  num_rows: 7403594
  total_columns: 30
  requested_columns: 30
  disk_mb: 111.9
  estimated_memory_mb: 1567.0
  fits_in_ram: True


2026-03-28 23:54:36,110 [INFO] Loaded 2024_Q1: 7403594 rows x 30 cols (1540.3 MB)
2026-03-28 23:54:38,226 [INFO] Sampled 2024_Q1: 7403594 -> 740359 rows
2026-03-28 23:54:38,226 [INFO] Sample created: 740359 rows (10.0%) -> data\bts\parquet\db1b_sample.parquet (26.0 MB)
2026-03-28 23:54:38,609 [INFO] Loaded sample: 740359 rows x 9 cols (23.4 MB)
C:\Users\LMT\AppData\Local\Temp\ipykernel_23468\109765115.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["Origin", "Dest"])["MktFare"]



Shape: (740359, 9)
Memory: 23.4 MB

MktFare stats:
count    740359.000000
mean        273.263330
std         226.746463
min           0.400000
25%         144.000000
50%         228.000000
75%         342.000000
max       12359.000000
Name: MktFare, dtype: float64

Top 10 routes:
                   mean  count
Origin Dest                   
JFK    LAX   652.501225    849
LAX    JFK   624.823804    786
ORD    LGA   211.754263    739
LAX    EWR   491.609767    729
EWR    LAX   520.082959    703
LGA    ORD   231.905758    693
LAX    HNL   333.787621    660
JFK    SFO   553.702074    651
HNL    LAX   353.232970    643
EWR    SFO   652.839666    628

Carriers: 25
Airports (origin): 422


2026-03-28 23:54:39,890 [INFO] Loaded 2024_Q1: 796349 rows x 7 cols (22.1 MB)
2026-03-28 23:54:39,890 [INFO] Route data: 796349 rows across 1 quarters | Origins: ['JFK', 'LAX', 'ORD', 'ATL', 'DFW'] | Destinations: all



Major hub data: 796349 rows


##### Existing Files Status View

In [14]:
# Check current state
status = handler.detect_pipeline_status(
    data_handler.Quarter(2024, 1),
    data_handler.Quarter(2025, 2),
)
handler.print_status(status)



Quarter         ZIP       CSV    Parquet   Valid         Rows Stage        Next Action
------------------------------------------------------------------------------------------
2024_Q1          --    1782MB      112MB     yes    7,403,594 parquet      process_features
2024_Q2          --    2055MB      129MB     yes    8,527,138 parquet      process_features
2024_Q3          --    1999MB      127MB     yes    8,290,377 parquet      process_features
2024_Q4          --    2051MB      129MB     yes    8,518,002 parquet      process_features
2025_Q1          --    1754MB      112MB     yes    7,290,528 parquet      process_features
2025_Q2          --    2035MB      130MB     yes    8,442,941 parquet      process_features

Summary: 6/6 quarters ready
  Total rows in Parquet: 48,472,580


##### Full Pipeline Runner

In [12]:
final = handler.run_pipeline(
    data_handler.Quarter(2024, 1),
    data_handler.Quarter(2025, 2),
    delete_csv=False,  # keep until verified
    delete_zip=True,   # reclaim space immediately
)

2026-03-28 23:15:13,442 [WARNING] Invalid Parquet file data\bts\parquet\db1b_market_2024_3.parquet: Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.
2026-03-28 23:15:13,444 [INFO] Pipeline status before run:



Quarter         ZIP       CSV    Parquet   Valid         Rows Stage        Next Action
------------------------------------------------------------------------------------------
2024_Q1          --    1782MB      112MB     yes    7,403,594 parquet      process_features
2024_Q2       105MB    2055MB         --      --           -- csv          convert_to_parquet
2024_Q3          --    1999MB       52MB    FAIL           -- parquet      reconvert
2024_Q4          --    2051MB         --      --           -- csv          convert_to_parquet
2025_Q1          --    1754MB         --      --           -- csv          convert_to_parquet
2025_Q2       105MB    2035MB         --      --           -- csv          convert_to_parquet

Summary: 1/6 quarters ready
  Needs conversion: 4
  Total rows in Parquet: 7,403,594


Pipeline progress:   0%|          | 0/5 [00:00<?, ?quarter/s]2026-03-28 23:15:13,444 [INFO] Converting 2024_Q2 (2055.5 MB CSV) -> Parquet
2026-03-28 23:16:34,701 [INFO] Conversion complete: 2024_Q2 | 8527138 rows | 2055.5 MB -> 129.4 MB (15.9x) | 81.3s (104941 rows/s)
Pipeline progress:  20%|██        | 1/5 [01:21<05:25, 81.31s/quarter]2026-03-28 23:16:34,758 [INFO] Converting 2024_Q3 (1999.3 MB CSV) -> Parquet
2026-03-28 23:17:59,066 [INFO] Conversion complete: 2024_Q3 | 8290377 rows | 1999.3 MB -> 126.8 MB (15.8x) | 84.3s (98335 rows/s)
Pipeline progress:  40%|████      | 2/5 [02:45<04:09, 83.08s/quarter]2026-03-28 23:17:59,083 [INFO] Converting 2024_Q4 (2051.1 MB CSV) -> Parquet
2026-03-28 23:19:46,179 [INFO] Conversion complete: 2024_Q4 | 8518002 rows | 2051.1 MB -> 128.6 MB (15.9x) | 107.1s (79538 rows/s)
Pipeline progress:  60%|██████    | 3/5 [04:32<03:08, 94.05s/quarter]2026-03-28 23:19:46,189 [INFO] Converting 2025_Q1 (1754.5 MB CSV) -> Parquet
2026-03-28 23:21:20,402 [INFO] C


Quarter         ZIP       CSV    Parquet   Valid         Rows Stage        Next Action
------------------------------------------------------------------------------------------
2024_Q1          --    1782MB      112MB     yes    7,403,594 parquet      process_features
2024_Q2          --    2055MB      129MB     yes    8,527,138 parquet      process_features
2024_Q3          --    1999MB      127MB     yes    8,290,377 parquet      process_features
2024_Q4          --    2051MB      129MB     yes    8,518,002 parquet      process_features
2025_Q1          --    1754MB      112MB     yes    7,290,528 parquet      process_features
2025_Q2          --    2035MB      130MB     yes    8,442,941 parquet      process_features

Summary: 6/6 quarters ready
  Total rows in Parquet: 48,472,580

PIPELINE RUN REPORT
  Duration:           7.9 minutes
  Processed:          5
  Skipped:            1
  Failed:             0
  Downloaded:         0.0 MB
  Rows converted:     41,068,986
  Parquet on di

##### Create sample of 10% for testing downstream components

In [7]:
sample_path = handler.create_sample(
    data_handler.Quarter(2024, 1),
    data_handler.Quarter(2025, 2),
    fraction=0.1,
)

2026-03-29 00:11:19,225 [INFO] Loaded 2024_Q1: 7403594 rows x 30 cols (1540.3 MB)
2026-03-29 00:11:21,706 [INFO] Sampled 2024_Q1: 7403594 -> 740359 rows
2026-03-29 00:11:42,499 [INFO] Loaded 2024_Q2: 8527138 rows x 30 cols (1774.9 MB)
2026-03-29 00:11:45,723 [INFO] Sampled 2024_Q2: 8527138 -> 852714 rows
2026-03-29 00:12:05,308 [INFO] Loaded 2024_Q3: 8290377 rows x 30 cols (1726.0 MB)
2026-03-29 00:12:08,196 [INFO] Sampled 2024_Q3: 8290377 -> 829038 rows
2026-03-29 00:12:27,423 [INFO] Loaded 2024_Q4: 8518002 rows x 30 cols (1771.9 MB)
2026-03-29 00:12:30,600 [INFO] Sampled 2024_Q4: 8518002 -> 851800 rows
2026-03-29 00:12:46,087 [INFO] Loaded 2025_Q1: 7290528 rows x 30 cols (1516.6 MB)
2026-03-29 00:12:48,276 [INFO] Sampled 2025_Q1: 7290528 -> 729053 rows
2026-03-29 00:13:05,933 [INFO] Loaded 2025_Q2: 8442941 rows x 30 cols (1757.1 MB)
2026-03-29 00:13:08,526 [INFO] Sampled 2025_Q2: 8442941 -> 844294 rows
2026-03-29 00:13:08,533 [INFO] Sample created: 4847258 rows (10.0%) -> data\bts\pa

In [11]:
df = handler.load_sample()
df

2026-03-29 00:15:29,171 [INFO] Loaded sample: 4847258 rows x 30 cols (1008.8 MB)


,ItinID,MktID,MktCoupons,Year,Quarter,OriginAirportID,Origin,OriginCountry,OriginState,DestAirportID,...,OpCarrier,BulkFare,Passengers,MktFare,MktDistance,MktDistanceGroup,MktMilesFlown,NonStopMiles,ItinGeoType,MktGeoType
0,202411970533,20241197053301,1,2024,1,11433,DTW,US,MI,13303,...,DL,0.0,4.0,373.0,1145.0,3,1145.0,1145.0,2,2
1,202412463889,20241246388902,1,2024,1,13487,MSP,US,MN,14635,...,DL,0.0,1.0,818.5,1416.0,3,1416.0,1416.0,2,2
2,202413353154,20241335315401,1,2024,1,14100,PHL,US,PA,12451,...,OH,0.0,1.0,605.5,742.0,2,742.0,742.0,2,2
3,202415072826,20241507282603,1,2024,1,10423,AUS,US,TX,11697,...,WN,0.0,1.0,288.0,1105.0,3,1105.0,1105.0,2,2
4,202411019257,20241101925703,2,2024,1,15027,STX,US,VI,14307,...,AA,0.0,1.0,7.5,2245.0,5,2245.0,1700.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4847253,202523190026,20252319002602,1,2025,2,11980,GRI,US,NE,10466,...,G4,0.0,7.0,21.0,905.0,2,905.0,905.0,2,2
4847254,202526935418,20252693541802,1,2025,2,12953,LGA,US,NY,15016,...,YX,0.0,1.0,284.0,888.0,2,888.0,888.0,2,2
4847255,202523603046,20252360304601,1,2025,2,10397,ATL,US,GA,11433,...,NK,0.0,5.0,65.0,594.0,2,594.0,594.0,2,2
4847256,202523838023,20252383802301,1,2025,2,11278,DCA,US,VA,15370,...,OH,0.0,1.0,363.0,1049.0,3,1049.0,1049.0,2,2
